<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/waveforms_to_fourier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Waveforms to Fourier Portraits
Vertical social video workflow with a separate install cell and a parametric render script.


# Project Documentation: Waveforms to Fourier Portraits

## Overview
This project provides an automated workflow for generating high-quality, vertical format educational videos that visualize the relationship between time-domain signals and their frequency-domain representations (Fourier Portraits). The script automates the rendering of mathematical functions, their corresponding Fourier transforms, and the final video assembly using FFmpeg.

## Theoretical Background

### The Fourier Transform
The Fourier Transform is a mathematical operation that decomposes a function of time (a signal) into the frequencies that make it up. This is analogous to how a musical chord can be expressed as the frequencies of its constituent notes.

The convention used in this project is the continuous-time Fourier transform:

$$X(f) = \int_{-\infty}^{\infty} x(t) e^{-i 2\pi ft} dt$$

### Time-Frequency Duality
A core concept illustrated in these visualizations is the duality between the time and frequency domains. Significant characteristics in one domain result in specific patterns in the other:

1.  **Symmetry and Smoothness**: Smooth, localized pulses like the Gaussian function remain smooth and localized in the frequency domain.
2.  **Sharp Transitions**: Functions with sharp edges or discontinuities in the time domain, such as the Rectangular pulse, result in high-frequency leakage and oscillations (sinc functions) in the frequency domain.
3.  **Modulation and Shifting**: Multiplying a signal by a cosine wave in the time domain (modulation) results in a shift of its spectrum in the frequency domain, appearing as two distinct peaks centered at the modulation frequency.
4.  **Scaling**: Compression in the time domain (a faster pulse) results in expansion in the frequency domain (a wider spectrum), reflecting the uncertainty principle of signal processing.

## Technical Implementation

### Rendering Engine
The visualization engine utilizes the Python Imaging Library (PIL/Pillow) to compose 1080x1920 (9:16) frames. Mathematical plotting is handled via NumPy and Matplotlib, which are then integrated into the PIL canvas.

### FFmpeg Pipeline
To handle the computational load of high-resolution video rendering, the script utilizes a segmented rendering approach. Frames are piped as raw bytes to FFmpeg subprocesses, which encode the segments using the H.264 codec. These segments are then concatenated losslessly to produce the final MP4 output. This method ensures memory efficiency and robust error handling during long render cycles.

In [2]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Waveforms to Fourier Portraits
GitHub: github.com/zombimann/Mathematical-video-animations-and-visualization
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, List, Tuple

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import subprocess
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties


# -----------------------------
# Parameter block
# -----------------------------
FPS = 18
CANVAS_W = 1080
CANVAS_H = 1920
DURATION_PER_SCENE = 5.0
CLOSING_DURATION = 1.8
OUTPUT_PATH = Path("/content/fourier_story_vertical.mp4")

BG = (11, 16, 32, 255)
CARD = (18, 26, 45, 255)
CARD_2 = (21, 31, 54, 255)
LINE = (54, 65, 89, 255)
TEXT = (245, 248, 255, 255)
MUTED = (190, 199, 214, 255)
SOFT = (140, 153, 178, 255)
ACCENT_LEFT = (125, 211, 252, 255)
ACCENT_RIGHT = (244, 114, 182, 255)
ACCENT_GOLD = (250, 204, 21, 255)
WATERMARK = (235, 240, 248, int(255 * 0.60))

TITLE = "Waveforms to Fourier Portraits"
FOURIER_CONVENTION_LABEL = "Fourier convention"
FOURIER_CONVENTION_FORMULA = r"$X(f)=\int x(t)e^{-i2\pi ft}dt$"


@dataclass(frozen=True)
class SceneSpec:
    name: str
    subtitle: str
    signal_formula: str
    spectrum_formula: str
    signal_fn: Callable[[np.ndarray], np.ndarray]
    spectrum_fn: Callable[[np.ndarray], np.ndarray]
    signal_range: Tuple[float, float]
    spectrum_range: Tuple[float, float]
    spectrum_note: str | None = None


# -----------------------------
# Math helpers
# -----------------------------

def smoothstep(x: float) -> float:
    x = min(1.0, max(0.0, x))
    return x * x * (3.0 - 2.0 * x)


def ease_in_out(x: float) -> float:
    return 0.5 - 0.5 * math.cos(math.pi * min(1.0, max(0.0, x)))


def normalized(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=float)
    m = np.max(np.abs(arr))
    return arr if m == 0 else arr / m


def sinc(u: np.ndarray) -> np.ndarray:
    return np.sinc(u)  # sin(pi u)/(pi u)


def rect(u: np.ndarray) -> np.ndarray:
    return (np.abs(u) <= 0.5).astype(float)


def tri(u: np.ndarray) -> np.ndarray:
    return np.maximum(1.0 - np.abs(u), 0.0)


# -----------------------------
# Signals and spectra
# -----------------------------

def gaussian_signal(t: np.ndarray, sigma: float = 0.55) -> np.ndarray:
    return np.exp(-(t ** 2) / (2 * sigma ** 2))


def gaussian_spectrum(f: np.ndarray, sigma: float = 0.55) -> np.ndarray:
    return np.exp(-2 * (math.pi ** 2) * (sigma ** 2) * (f ** 2))


def rect_signal(t: np.ndarray, width: float = 1.25) -> np.ndarray:
    return rect(t / width)


def rect_spectrum(f: np.ndarray, width: float = 1.25) -> np.ndarray:
    return np.abs(sinc(width * f))


def tri_signal(t: np.ndarray, width: float = 1.15) -> np.ndarray:
    return tri(t / width)


def tri_spectrum(f: np.ndarray, width: float = 1.15) -> np.ndarray:
    return sinc(width * f) ** 2


def damped_cosine_signal(t: np.ndarray, a: float = 1.45, f0: float = 1.65) -> np.ndarray:
    return np.exp(-a * np.abs(t)) * np.cos(2 * math.pi * f0 * t)


def damped_cosine_spectrum(f: np.ndarray, a: float = 1.45, f0: float = 1.65) -> np.ndarray:
    left = a / (a * a + (2 * math.pi * (f - f0)) ** 2)
    right = a / (a * a + (2 * math.pi * (f + f0)) ** 2)
    return left + right


def double_gaussian_signal(t: np.ndarray, sigma: float = 0.32, d: float = 1.08) -> np.ndarray:
    return np.exp(-((t - d) ** 2) / (2 * sigma ** 2)) + np.exp(-((t + d) ** 2) / (2 * sigma ** 2))


def double_gaussian_spectrum(f: np.ndarray, sigma: float = 0.32, d: float = 1.08) -> np.ndarray:
    envelope = np.exp(-2 * (math.pi ** 2) * (sigma ** 2) * (f ** 2))
    return envelope * np.abs(np.cos(2 * math.pi * d * f))


def derivative_gaussian_signal(t: np.ndarray, sigma: float = 0.58) -> np.ndarray:
    return -(t / (sigma ** 2)) * np.exp(-(t ** 2) / (2 * sigma ** 2))


def derivative_gaussian_spectrum(f: np.ndarray, sigma: float = 0.58) -> np.ndarray:
    return np.abs(f) * np.exp(-2 * (math.pi ** 2) * (sigma ** 2) * (f ** 2))


SCENES: List[SceneSpec] = [
    SceneSpec(
        name="Gaussian pulse",
        subtitle="A smooth pulse stays smooth in frequency.",
        signal_formula=r"$x(t)=e^{-t^2/(2\sigma^2)}$",
        spectrum_formula=r"$|X(f)|=e^{-2\pi^2\sigma^2 f^2}$",
        signal_fn=lambda t: gaussian_signal(t, 0.55),
        spectrum_fn=lambda f: gaussian_spectrum(f, 0.55),
        signal_range=(-2.0, 2.0),
        spectrum_range=(-3.5, 3.5),
    ),
    SceneSpec(
        name="Rectangular pulse",
        subtitle="A sharp edge becomes ripples.",
        signal_formula=r"$x(t)=\operatorname{rect}(t/T)$",
        spectrum_formula=r"$|X(f)|=T\,|\operatorname{sinc}(Tf)|$",
        signal_fn=lambda t: rect_signal(t, 1.25),
        spectrum_fn=lambda f: rect_spectrum(f, 1.25),
        signal_range=(-2.2, 2.2),
        spectrum_range=(-4.0, 4.0),
    ),
    SceneSpec(
        name="Triangular pulse",
        subtitle="A gentler edge narrows the ripple story.",
        signal_formula=r"$x(t)=\operatorname{tri}(t/T)$",
        spectrum_formula=r"$|X(f)|=T\,\operatorname{sinc}^2(Tf)$",
        signal_fn=lambda t: tri_signal(t, 1.15),
        spectrum_fn=lambda f: tri_spectrum(f, 1.15),
        signal_range=(-2.0, 2.0),
        spectrum_range=(-4.0, 4.0),
    ),
    SceneSpec(
        name="Damped cosine",
        subtitle="One note, softened by decay, makes twin hills.",
        signal_formula=r"$x(t)=e^{-a|t|}\cos(2\pi f_0 t)$",
        spectrum_formula=r"$|X(f)|\propto L(f-f_0)+L(f+f_0)$",
        signal_fn=lambda t: damped_cosine_signal(t, 1.45, 1.65),
        spectrum_fn=lambda f: damped_cosine_spectrum(f, 1.45, 1.65),
        signal_range=(-2.2, 2.2),
        spectrum_range=(-4.0, 4.0),
        spectrum_note=r"$L(u)=\frac{1}{a^2+(2\pi u)^2}$",
    ),
    SceneSpec(
        name="Two pulses",
        subtitle="Two arrivals create interference fringes.",
        signal_formula=r"$x(t)=g(t-d)+g(t+d)$",
        spectrum_formula=r"$|X(f)|=2G(f)\,|\cos(2\pi d f)|$",
        signal_fn=lambda t: double_gaussian_signal(t, 0.32, 1.08),
        spectrum_fn=lambda f: double_gaussian_spectrum(f, 0.32, 1.08),
        signal_range=(-2.5, 2.5),
        spectrum_range=(-4.0, 4.0),
    ),
    SceneSpec(
        name="Derivative of Gaussian",
        subtitle="Fast change pushes energy outward.",
        signal_formula=r"$x(t)=-\frac{t}{\sigma^2}e^{-t^2/(2\sigma^2)}$",
        spectrum_formula=r"$|X(f)|\propto |f|\,e^{-2\pi^2\sigma^2 f^2}$",
        signal_fn=lambda t: derivative_gaussian_signal(t, 0.58),
        spectrum_fn=lambda f: derivative_gaussian_spectrum(f, 0.58),
        signal_range=(-2.2, 2.2),
        spectrum_range=(-4.0, 4.0),
    ),
]

SCENE_BASES: list[Image.Image] = []
SCENE_PLOTS: list[dict[str, np.ndarray]] = []
for _spec in SCENES:
    SCENE_BASES.append(None)  # placeholder
    signal_x = np.linspace(_spec.signal_range[0], _spec.signal_range[1], 1400)
    spectrum_x = np.linspace(_spec.spectrum_range[0], _spec.spectrum_range[1], 1400)
    SCENE_PLOTS.append({
        "signal_x": signal_x,
        "signal_y": normalized(_spec.signal_fn(signal_x)),
        "spectrum_x": spectrum_x,
        "spectrum_y": normalized(_spec.spectrum_fn(spectrum_x)),
    })


# -----------------------------
# Fonts and rendered text assets
# -----------------------------
FONT_REGULAR = font_manager.findfont(FontProperties(family="DejaVu Sans", weight="normal"))
FONT_BOLD = font_manager.findfont(FontProperties(family="DejaVu Sans", weight="bold"))
TITLE_FONT = ImageFont.truetype(FONT_BOLD, 44)
SUBTITLE_FONT = ImageFont.truetype(FONT_REGULAR, 24)
PANEL_FONT = ImageFont.truetype(FONT_BOLD, 22)
SMALL_FONT = ImageFont.truetype(FONT_REGULAR, 20)
WATERMARK_FONT = ImageFont.truetype(FONT_REGULAR, 24)
CLOSING_FONT = ImageFont.truetype(FONT_BOLD, 46)
CLOSING_SMALL = ImageFont.truetype(FONT_REGULAR, 36)


def render_math_formula(formula: str, fontsize: int = 36, dpi: int = 220) -> Image.Image:
    """Render a mathtext string into a transparent RGBA image."""
    fig = plt.figure(figsize=(0.01, 0.01), dpi=dpi)
    fig.patch.set_alpha(0)
    plt.axis("off")
    text = plt.text(0, 0, formula, fontsize=fontsize, color="white")
    fig.canvas.draw()
    bbox = text.get_window_extent(renderer=fig.canvas.get_renderer())
    width = int(bbox.width / 100.0 * dpi) + 30
    height = int(bbox.height / 100.0 * dpi) + 30
    plt.close(fig)

    fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)
    fig.patch.set_alpha(0)
    plt.axis("off")
    plt.text(0, 0, formula, fontsize=fontsize, color="white")
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.tostring_argb(), dtype=np.uint8)
    buf.shape = (fig.canvas.get_width_height()[::-1] + (4,))
    buf = buf[:, :, [1, 2, 3, 0]]
    plt.close(fig)
    return Image.fromarray(buf, mode="RGBA")


def text_box(text: str, font: ImageFont.FreeTypeFont, fill=TEXT) -> Image.Image:
    dummy = Image.new("RGBA", (10, 10), (0, 0, 0, 0))
    d = ImageDraw.Draw(dummy)
    bbox = d.textbbox((0, 0), text, font=font)
    w = bbox[2] - bbox[0] + 8
    h = bbox[3] - bbox[1] + 8
    img = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)
    d.text((4, 0), text, font=font, fill=fill)
    return img


def fit_image(img: Image.Image, max_width: int, max_height: int | None = None) -> Image.Image:
    w, h = img.size
    scale = min(max_width / w, (max_height / h) if max_height else 1.0, 1.0)
    if scale >= 1.0:
        return img
    new_size = (max(1, int(w * scale)), max(1, int(h * scale)))
    return img.resize(new_size, Image.Resampling.LANCZOS)


def draw_rounded_rect(draw: ImageDraw.ImageDraw, xy, radius, fill, outline=None, width=1):
    draw.rounded_rectangle(xy, radius=radius, fill=fill, outline=outline, width=width)


# -----------------------------
# Layout and plotting utilities
# -----------------------------
LEFT_X = 60
RIGHT_X = 555
CARD_Y = 315
CARD_W = 465
CARD_H = 1410
CARD_RADIUS = 34
PLOT_MARGIN_X = 26
PLOT_TOP = 100
PLOT_H = 835
FORMULA_TOP = 985
FORMULA_H = 305

PLOT_LEFT = (LEFT_X + PLOT_MARGIN_X, CARD_Y + PLOT_TOP, LEFT_X + CARD_W - PLOT_MARGIN_X, CARD_Y + PLOT_TOP + PLOT_H)
PLOT_RIGHT = (RIGHT_X + PLOT_MARGIN_X, CARD_Y + PLOT_TOP, RIGHT_X + CARD_W - PLOT_MARGIN_X, CARD_Y + PLOT_TOP + PLOT_H)


def data_to_pixel(x: np.ndarray, y: np.ndarray, bounds: Tuple[float, float], rect_box) -> Tuple[np.ndarray, np.ndarray]:
    x0, y0, x1, y1 = rect_box
    xmin, xmax = bounds
    ymin, ymax = -1.18, 1.18
    xp = x0 + (x - xmin) / (xmax - xmin) * (x1 - x0)
    yp = y1 - (y - ymin) / (ymax - ymin) * (y1 - y0)
    return xp, yp


def draw_axes(base: ImageDraw.ImageDraw, rect_box, x_label: str, tone: Tuple[int, int, int, int]):
    x0, y0, x1, y1 = rect_box
    # Grid
    for frac in np.linspace(0.0, 1.0, 6):
        x = x0 + frac * (x1 - x0)
        base.line((x, y0, x, y1), fill=(50, 59, 79, 120), width=1)
    for frac in np.linspace(0.0, 1.0, 5):
        y = y0 + frac * (y1 - y0)
        base.line((x0, y, x1, y), fill=(50, 59, 79, 110), width=1)
    # Axes
    mid_y = y1 - (0 - (-1.18)) / (1.18 - (-1.18)) * (y1 - y0)
    base.line((x0, mid_y, x1, mid_y), fill=(130, 143, 172, 160), width=2)
    # Tick labels
    for val in [-1, 0, 1]:
        y = y1 - (val - (-1.18)) / (1.18 - (-1.18)) * (y1 - y0)
        base.line((x0 - 8, y, x0 + 8, y), fill=(190, 199, 214, 180), width=2)
        base.text((x0 - 44, y - 14), f"{val:+d}".replace("+0", "0"), font=SMALL_FONT, fill=SOFT)
    base.text((x1 - 20, mid_y + 10), x_label, font=SMALL_FONT, fill=tone)


def plot_curve(draw: ImageDraw.ImageDraw, rect_box, x: np.ndarray, y: np.ndarray, bounds: Tuple[float, float], color, width=8, alpha=255):
    xp, yp = data_to_pixel(x, y, bounds, rect_box)
    pts = list(zip(xp.tolist(), yp.tolist()))
    overlay = Image.new("RGBA", (CANVAS_W, CANVAS_H), (0, 0, 0, 0))
    od = ImageDraw.Draw(overlay)
    c = color[:3] + (alpha,)
    if len(pts) > 1:
        od.line(pts, fill=c, width=width, joint="curve")
    return overlay


def plot_fill(rect_box, x: np.ndarray, y: np.ndarray, bounds: Tuple[float, float], color, alpha=40):
    x0, y0, x1, y1 = rect_box
    xp, yp = data_to_pixel(x, y, bounds, rect_box)
    baseline = np.full_like(xp, y1)
    pts = list(zip(xp.tolist(), yp.tolist())) + list(zip(xp[::-1].tolist(), baseline[::-1].tolist()))
    overlay = Image.new("RGBA", (CANVAS_W, CANVAS_H), (0, 0, 0, 0))
    od = ImageDraw.Draw(overlay)
    od.polygon(pts, fill=color[:3] + (alpha,))
    return overlay


def draw_marker(base: ImageDraw.ImageDraw, rect_box, x_value: float, y_value: float, bounds: Tuple[float, float], color):
    xp, yp = data_to_pixel(np.array([x_value]), np.array([y_value]), bounds, rect_box)
    x, y = float(xp[0]), float(yp[0])
    base.ellipse((x - 10, y - 10, x + 10, y + 10), fill=color)
    base.ellipse((x - 18, y - 18, x + 18, y + 18), outline=color[:3] + (120, ), width=2)


def draw_vertical_scan(base: ImageDraw.ImageDraw, rect_box, x_value: float, bounds: Tuple[float, float], color, alpha=80):
    x0, y0, x1, y1 = rect_box
    xmin, xmax = bounds
    xp = x0 + (x_value - xmin) / (xmax - xmin) * (x1 - x0)
    base.line((xp, y0, xp, y1), fill=color[:3] + (alpha,), width=3)


# -----------------------------
# Compositing helpers
# -----------------------------

def draw_watermark(base: Image.Image):
    d = ImageDraw.Draw(base)
    text = "© Mugambi Ndwiga / @craftsandengineering"
    bbox = d.textbbox((0, 0), text, font=WATERMARK_FONT)
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = CANVAS_W - w - 44
    y = CANVAS_H - h - 32
    d.text((x, y), text, font=WATERMARK_FONT, fill=WATERMARK)


def make_scene_base(spec: SceneSpec) -> Image.Image:
    img = Image.new("RGBA", (CANVAS_W, CANVAS_H), BG)
    d = ImageDraw.Draw(img)

    # Title band
    d.rounded_rectangle((42, 42, CANVAS_W - 42, 242), radius=40, fill=(13, 20, 38, 255), outline=(33, 49, 84, 255), width=2)
    d.line((90, 208, CANVAS_W - 90, 208), fill=(72, 84, 114, 200), width=2)

    title_img = text_box(TITLE, TITLE_FONT, TEXT)
    img.alpha_composite(title_img, (int((CANVAS_W - title_img.width) / 2), 52))

    subtitle = text_box(spec.subtitle, SUBTITLE_FONT, TEXT)
    img.alpha_composite(subtitle, (int((CANVAS_W - subtitle.width) / 2), 110))

    conv_label = text_box(FOURIER_CONVENTION_LABEL, SMALL_FONT, MUTED)
    img.alpha_composite(conv_label, (int((CANVAS_W - conv_label.width) / 2), 143))
    conv_img = fit_image(render_math_formula(FOURIER_CONVENTION_FORMULA, fontsize=16), 520)
    img.alpha_composite(conv_img, (int((CANVAS_W - conv_img.width) / 2), 164))

    # Cards
    for x in [LEFT_X, RIGHT_X]:
        draw_rounded_rect(d, (x, CARD_Y, x + CARD_W, CARD_Y + CARD_H), CARD_RADIUS, CARD, outline=(37, 53, 88, 255), width=2)
        d.rounded_rectangle((x + 18, CARD_Y + 18, x + CARD_W - 18, CARD_Y + 76), radius=22, fill=CARD_2, outline=(52, 65, 94, 255), width=1)

    d.text((LEFT_X + 34, CARD_Y + 30), "TIME DOMAIN", font=PANEL_FONT, fill=ACCENT_LEFT)
    d.text((RIGHT_X + 34, CARD_Y + 30), "FREQUENCY DOMAIN", font=PANEL_FONT, fill=ACCENT_RIGHT)

    draw_axes(d, PLOT_LEFT, "t", ACCENT_LEFT)
    draw_axes(d, PLOT_RIGHT, "f", ACCENT_RIGHT)

    # Card separators
    d.line((LEFT_X + 24, CARD_Y + FORMULA_TOP - 18, LEFT_X + CARD_W - 24, CARD_Y + FORMULA_TOP - 18), fill=(62, 74, 98, 180), width=2)
    d.line((RIGHT_X + 24, CARD_Y + FORMULA_TOP - 18, RIGHT_X + CARD_W - 24, CARD_Y + FORMULA_TOP - 18), fill=(62, 74, 98, 180), width=2)

    # Formula placeholders via rendered images
    sig_img = fit_image(render_math_formula(spec.signal_formula, fontsize=20), CARD_W - 70)
    spc_img = fit_image(render_math_formula(spec.spectrum_formula, fontsize=20), CARD_W - 70)
    img.alpha_composite(sig_img, (LEFT_X + int((CARD_W - sig_img.width) / 2), CARD_Y + FORMULA_TOP + 54))
    img.alpha_composite(spc_img, (RIGHT_X + int((CARD_W - spc_img.width) / 2), CARD_Y + FORMULA_TOP + 42))
    if spec.spectrum_note:
        note_img = fit_image(render_math_formula(spec.spectrum_note, fontsize=18), CARD_W - 80)
        img.alpha_composite(note_img, (RIGHT_X + int((CARD_W - note_img.width) / 2), CARD_Y + FORMULA_TOP + 92))

    # Small captions
    cap1 = text_box("signal shape", SMALL_FONT, MUTED)
    cap2 = text_box("spectrum shape", SMALL_FONT, MUTED)
    img.alpha_composite(cap1, (LEFT_X + 34, CARD_Y + FORMULA_TOP - 52))
    img.alpha_composite(cap2, (RIGHT_X + 34, CARD_Y + FORMULA_TOP - 52))

    draw_watermark(img)
    return img

for _i, _spec in enumerate(SCENES):
    SCENE_BASES[_i] = make_scene_base(_spec)


def make_frame(spec: SceneSpec, t: float, scene_index: int, scene_duration: float) -> np.ndarray:
    base = SCENE_BASES[scene_index].copy()
    d = ImageDraw.Draw(base)

    # Scene progress with gentle lead-in/out.
    p = min(1.0, max(0.0, t / scene_duration))
    fade_in = smoothstep(p / 0.18)
    fade_out = smoothstep((1 - p) / 0.14)
    alpha = int(255 * min(fade_in, fade_out, 1.0))

    # Data
    plot_data = SCENE_PLOTS[scene_index]
    signal_x = plot_data["signal_x"]
    signal_y = plot_data["signal_y"]
    spectrum_x = plot_data["spectrum_x"]
    spectrum_y = plot_data["spectrum_y"]

    # Add subtle baseline fill.
    overlay = Image.new("RGBA", (CANVAS_W, CANVAS_H), (0, 0, 0, 0))
    overlay = Image.alpha_composite(overlay, plot_fill(PLOT_LEFT, signal_x, signal_y, spec.signal_range, ACCENT_LEFT, alpha=26))
    overlay = Image.alpha_composite(overlay, plot_fill(PLOT_RIGHT, spectrum_x, spectrum_y, spec.spectrum_range, ACCENT_RIGHT, alpha=24))
    base = Image.alpha_composite(base, overlay)
    d = ImageDraw.Draw(base)

    # Curves.
    curve_opacity = int(235 * alpha / 255)
    curve_left = plot_curve(d, PLOT_LEFT, signal_x, signal_y, spec.signal_range, ACCENT_LEFT, width=8, alpha=curve_opacity)
    curve_right = plot_curve(d, PLOT_RIGHT, spectrum_x, spectrum_y, spec.spectrum_range, ACCENT_RIGHT, width=8, alpha=curve_opacity)
    base = Image.alpha_composite(base, curve_left)
    base = Image.alpha_composite(base, curve_right)
    d = ImageDraw.Draw(base)

    # A traveling cursor on the signal side.
    cursor_x = spec.signal_range[0] + (spec.signal_range[1] - spec.signal_range[0]) * ease_in_out(p)
    cursor_y = float(np.interp(cursor_x, signal_x, signal_y))
    draw_vertical_scan(d, PLOT_LEFT, cursor_x, spec.signal_range, ACCENT_GOLD, alpha=95)
    draw_marker(d, PLOT_LEFT, cursor_x, cursor_y, spec.signal_range, ACCENT_GOLD)

    # Frequency-side soft scan line that sweeps more slowly.
    freq_cursor = spec.spectrum_range[0] + (spec.spectrum_range[1] - spec.spectrum_range[0]) * ease_in_out((p * 0.8 + 0.12) % 1.0)
    draw_vertical_scan(d, PLOT_RIGHT, freq_cursor, spec.spectrum_range, ACCENT_GOLD, alpha=72)

    # Add tiny descriptor on the lower margin for experts.
    expert_note = text_box("left: x(t)   right: |X(f)| ", SMALL_FONT, SOFT)
    base.alpha_composite(expert_note, (int((CANVAS_W - expert_note.width) / 2), 252))

    draw_watermark(base)
    return np.array(base.convert("RGB"))


def make_closing_frame(t: float) -> np.ndarray:
    img = Image.new("RGBA", (CANVAS_W, CANVAS_H), BG)
    d = ImageDraw.Draw(img)

    # Solid title card with the same primary color family.
    d.rounded_rectangle((60, 420, CANVAS_W - 60, 1360), radius=48, fill=(14, 22, 40, 255), outline=(32, 49, 83, 255), width=2)
    title = text_box("Made by Mugambi Ndwiga", CLOSING_FONT, TEXT)
    handle = text_box("@craftsandengineering", CLOSING_SMALL, TEXT)
    img.alpha_composite(title, (int((CANVAS_W - title.width) / 2), 760))
    img.alpha_composite(handle, (int((CANVAS_W - handle.width) / 2), 840))
    draw_watermark(img)
    return np.array(img.convert("RGB"))


# -----------------------------
# Render
# -----------------------------

def render_segment(start_t: float, end_t: float, output_path: Path) -> None:
    scene_duration = DURATION_PER_SCENE
    total_main = scene_duration * len(SCENES)
    total_frames = int(round((end_t - start_t) * FPS))

    ffmpeg_cmd = [
        "ffmpeg", "-y",
        "-f", "rawvideo",
        "-vcodec", "rawvideo",
        "-pix_fmt", "rgb24",
        "-s", f"{CANVAS_W}x{CANVAS_H}",
        "-r", str(FPS),
        "-i", "-",
        "-an",
        "-c:v", "libx264",
        "-preset", "ultrafast",
        "-crf", "30",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(output_path),
    ]

    proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)
    try:
        for frame_idx in range(total_frames):
            t = start_t + frame_idx / FPS
            if t >= total_main:
                frame = make_closing_frame(t - total_main)
            else:
                scene_index = min(len(SCENES) - 1, int(t // scene_duration))
                local_t = t - scene_index * scene_duration
                frame = make_frame(SCENES[scene_index], local_t, scene_index, scene_duration)
            assert proc.stdin is not None
            proc.stdin.write(frame.tobytes())
        assert proc.stdin is not None
        proc.stdin.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f"ffmpeg failed with code {ret} for segment {start_t:.2f}-{end_t:.2f}")
    finally:
        if proc.stdin and not proc.stdin.closed:
            proc.stdin.close()


def render_video(output_path: Path = OUTPUT_PATH, segment_seconds: float = 4.0) -> None:
    import tempfile

    scene_duration = DURATION_PER_SCENE
    total_main = scene_duration * len(SCENES)
    total_duration = total_main + CLOSING_DURATION
    segment_paths: list[Path] = []

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)
        start = 0.0
        idx = 0
        while start < total_duration - 1e-6:
            end = min(total_duration, start + segment_seconds)
            segment_path = tmpdir_path / f"segment_{idx:02d}.mp4"
            render_segment(start, end, segment_path)
            segment_paths.append(segment_path)
            start = end
            idx += 1

        concat_list = tmpdir_path / "concat.txt"
        concat_list.write_text("\n".join(f"file '{p.as_posix()}'" for p in segment_paths) + "\n")
        ffmpeg_concat = [
            "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(concat_list),
            "-c", "copy", str(output_path)
        ]
        subprocess.run(ffmpeg_concat, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


if __name__ == "__main__":
    render_video()

/tmp/ipykernel_3035/3503373426.py:262: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(buf, mode="RGBA")


In [3]:
from IPython.display import Video

# Display the rendered video
Video('/content/fourier_story_vertical.mp4', embed=True, width=400)